In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into memory (store full tables in variables)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to convert epoch seconds to tz-aware ISO string
def epoch_to_tz_iso(epoch):
    if pd.isna(epoch):
        return None
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).isoformat()

# Compact summary builder for telemetry files
def build_summary(df, id_col, name_col):
    total_rows = len(df)
    min_ts = None if total_rows == 0 else int(df['timestamp'].min())
    max_ts = None if total_rows == 0 else int(df['timestamp'].max())
    cmdb_list = list(pd.Series(df[id_col].dropna().unique())[:50])
    name_list = list(pd.Series(df[name_col].dropna().unique())[:50])
    return {
        'total_rows': total_rows,
        'min_timestamp': min_ts,
        'min_timestamp_tz': epoch_to_tz_iso(min_ts) if min_ts is not None else None,
        'max_timestamp': max_ts,
        'max_timestamp_tz': epoch_to_tz_iso(max_ts) if max_ts is not None else None,
        'unique_cmdb_ids_sample (<=50)': cmdb_list,
        'unique_names_sample (<=50)': name_list
    }

# Summaries for each file
summary_metric = build_summary(df_metric, 'cmdb_id', 'kpi_name')
summary_trace = build_summary(df_trace, 'cmdb_id', 'trace_name')
summary_log = build_summary(df_log, 'cmdb_id', 'log_name')

# Compute per-(cmdb_id,kpi_name) count and global P95 for metric
if not df_metric.empty:
    metric_group = (
        df_metric
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .agg(count='count', p95_value=lambda s: float(s.quantile(0.95)))
        .reset_index()
        .sort_values('count', ascending=False)
    )
    metric_top20 = metric_group[['cmdb_id', 'kpi_name', 'count', 'p95_value']].head(20)
else:
    metric_top20 = pd.DataFrame(columns=['cmdb_id', 'kpi_name', 'count', 'p95_value'])

# Compute per-(cmdb_id,trace_name) for trace
if not df_trace.empty:
    trace_group = (
        df_trace
        .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
        .agg(count='count', p95_value=lambda s: float(s.quantile(0.95)))
        .reset_index()
        .sort_values('count', ascending=False)
    )
    trace_top20 = trace_group[['cmdb_id', 'trace_name', 'count', 'p95_value']].head(20)
else:
    trace_top20 = pd.DataFrame(columns=['cmdb_id', 'trace_name', 'count', 'p95_value'])

# Compute per-(cmdb_id,log_name) for log
if not df_log.empty:
    log_group = (
        df_log
        .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
        .agg(count='count', p95_value=lambda s: float(s.quantile(0.95)))
        .reset_index()
        .sort_values('count', ascending=False)
    )
    log_top20 = log_group[['cmdb_id', 'log_name', 'count', 'p95_value']].head(20)
else:
    log_top20 = pd.DataFrame(columns=['cmdb_id', 'log_name', 'count', 'p95_value'])

# Error logs summary: total count and up to first 20 rows (timestamp, cmdb_id, message)
error_count = len(df_error)
if error_count > 0:
    # also add a tz-aware timestamp string column for readability but keep numeric timestamp too
    df_error_head = df_error[['timestamp', 'cmdb_id', 'message']].head(20).copy()
    df_error_head['timestamp_tz'] = df_error_head['timestamp'].apply(lambda x: epoch_to_tz_iso(int(x)) if pd.notna(x) else None)
else:
    df_error_head = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message', 'timestamp_tz'])

# Final compact outputs (variables to display)
summary_metric, metric_top20, summary_trace, trace_top20, summary_log, log_top20, error_count, df_error_head

```
Out[1]:
```
```python
# Compose a concise plain-English summary using existing variables from the kernel
metric_s = summary_metric
trace_s = summary_trace
log_s = summary_log

def top_lines(df, cols, n=3):
    lines = []
    for _, r in df.head(n).iterrows():
        vals = [str(r[c]) for c in cols]
        lines.append(" | ".join(vals))
    return lines

metric_top_lines = top_lines(metric_top20, ['cmdb_id','kpi_name','count','p95_value'], 3) if not metric_top20.empty else []
trace_top_lines = top_lines(trace_top20, ['cmdb_id','trace_name','count','p95_value'], 3) if not trace_top20.empty else []
log_top_lines = top_lines(log_top20, ['cmdb_id','log_name','count','p95_value'], 3) if not log_top20.empty else []

summary_text = (
    "Summary of telemetry files (all times shown in UTC+8):\n\n"
    f"1) metric.csv: {metric_s['total_rows']} rows, time range {metric_s['min_timestamp_tz']} to {metric_s['max_timestamp_tz']}.\n"
    f"   Sample CMDB IDs: {', '.join(map(str, metric_s['unique_cmdb_ids_sample (<=50)'][:10]))} (and more).\n"
    f"   Sample KPI names: {', '.join(map(str, metric_s['unique_names_sample (<=50)'][:6]))} (and more).\n"
    "   Top (cmdb_id, kpi_name) by point count (showing up to 3):\n"
    + ("\n".join(f"     - {l}" for l in metric_top_lines) if metric_top_lines else "     - (none)") + "\n\n"
    f"2) trace.csv: {trace_s['total_rows']} rows, time range {trace_s['min_timestamp_tz']} to {trace_s['max_timestamp_tz']}.\n"
    f"   Sample CMDB IDs: {', '.join(map(str, trace_s['unique_cmdb_ids_sample (<=50)'][:8]))} (and more).\n"
    f"   Sample trace names: {', '.join(map(str, trace_s['unique_names_sample (<=50)'][:6]))} (and more).\n"
    "   Top (cmdb_id, trace_name) by point count (showing up to 3):\n"
    + ("\n".join(f"     - {l}" for l in trace_top_lines) if trace_top_lines else "     - (none)") + "\n\n"
    f"3) log.csv: {log_s['total_rows']} rows, time range {log_s['min_timestamp_tz']} to {log_s['max_timestamp_tz']}.\n"
    f"   Sample CMDB IDs: {', '.join(map(str, log_s['unique_cmdb_ids_sample (<=50)'][:8]))} (and more).\n"
    f"   Sample log names: {', '.join(map(str, log_s['unique_names_sample (<=50)'][:6]))} (and more).\n"
    "   Top (cmdb_id, log_name) by point count (showing up to 3):\n"
    + ("\n".join(f"     - {l}" for l in log_top_lines) if log_top_lines else "     - (none)") + "\n\n"
    f"4) error_logs.csv: {error_count} rows total. First up to 20 error rows were returned; they show many apache requests (e.g., POST to /UOCP/*) around 2021-03-12T22:35 to 22:36+08:00.\n\n"
    "Notes:\n"
    "- Many top metric/trace/log series have ~30–31 points (consistent short time window).\n"
    "- Trace top series (e.g., IG01 trace.self.duration_mean) show p95s in seconds (examples above).\n"
    "- Apache and Tomcat components appear frequently in logs and error logs.\n"
    "This is a compact overview; let me know if you want CSV-specific details (full top-20 tables, per-series stats, or time-sliced analysis)."
)

# Display the summary string (as required: use variable rather than print)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

({'total_rows': 27204, 'min_timestamp': 1615559400, 'min_timestamp_tz': '2021-03-12T22:30:00+08:00', 'max_timestamp': 1615561200, 'max_timestamp_tz': '2021-03-12T23:00:00+08:00', 'unique_cmdb_ids_sample (<=50)': ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA2', 'dockerB1', 'dockerA1', 'dockerB2'], 'unique_names_sample (<=50)': ['JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'JVM-Threads_7778_JVM_ThreadCount_Threads', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies', 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount', 'OSLinux-OSLinux_SYSTEM_SYSTEM_Check-DefaultRoute', 'OSLinux-OSLinux_ZABBIX_Host_Uptime', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_FileSizeMB', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvgServ', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvgServ', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPercentBusy', 'OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec', 'OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc', 'OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent', 'OSLinux-OSLinux_SWAP_SWAP_Si', 'OSLinux-OSLinux_SWAP_SWAP_So', 'OSLinux-OSLinux_SYSTEM_SYSTEM_Check-Hostname', 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-1_SingleCpuidle', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-OSLinux_FILESYSTEM_-_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPercent']},            cmdb_id                                           kpi_name  count     p95_value
1611      apache02                             OSLinux-CPU_CPU_CPUWio     31  1.516610e-01
6             IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  7.350000e+01
1174      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  9.870000e+02
1175      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  4.053960e+05
1176      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  1.579615e+09
1177      Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  2.429628e+06
1178      Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31  0.000000e+00
1179      Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31  0.000000e+00
1180      Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31  0.000000e+00
1181      Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter     31  0.000000e+00
1182      Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00
1183      Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00
1081  ServiceTest9                                             app.rr     31  1.000000e+02
1082  ServiceTest9                                             app.sr     31  1.000000e+02
1072  ServiceTest7                                            app.mrt     31  9.409740e+00
1073  ServiceTest7                                             app.rr     31  1.000000e+02
1074  ServiceTest7                                             app.sr     31  1.000000e+02
1076  ServiceTest8                                            app.mrt     31  7.588135e+00
1077  ServiceTest8                                             app.rr     31  1.000000e+02
1078  ServiceTest8                                             app.sr     31  1.000000e+02, {'total_rows': 6852, 'min_timestamp': 1615559400, 'min_timestamp_tz': '2021-03-12T22:30:00+08:00', 'max_timestamp': 1615561140, 'max_timestamp_tz': '2021-03-12T22:59:00+08:00', 'unique_cmdb_ids_sample (<=50)': ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 'unique_names_sample (<=50)': ['trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_Tomcat01.duration_mean', 'trace.to_Tomcat01.duration_p95', 'trace.to_Tomcat01.row_count', 'trace.to_Tomcat02.duration_mean', 'trace.to_Tomcat02.duration_p95', 'trace.to_Tomcat02.row_count', 'trace.to_Tomcat03.duration_mean', 'trace.to_Tomcat03.duration_p95', 'trace.to_Tomcat03.row_count', 'trace.to_Tomcat04.duration_mean', 'trace.to_Tomcat04.duration_p95', 'trace.to_Tomcat04.row_count', 'trace.from_Tomcat01.duration_mean', 'trace.from_Tomcat01.duration_p95', 'trace.from_Tomcat01.row_count', 'trace.from_Tomcat02.duration_mean', 'trace.from_Tomcat02.duration_p95', 'trace.from_Tomcat02.row_count', 'trace.from_Tomcat03.duration_mean', 'trace.from_Tomcat03.duration_p95', 'trace.from_Tomcat03.row_count', 'trace.from_Tomcat04.duration_mean', 'trace.from_Tomcat04.duration_p95', 'trace.from_Tomcat04.row_count', 'trace.from_dockerA1.duration_mean', 'trace.from_dockerA1.duration_p95', 'trace.from_dockerA1.row_count', 'trace.from_dockerA2.duration_mean', 'trace.from_dockerA2.duration_p95', 'trace.from_dockerA2.row_count', 'trace.from_dockerB1.duration_mean', 'trace.from_dockerB1.duration_p95', 'trace.from_dockerB1.row_count', 'trace.from_dockerB2.duration_mean', 'trace.from_dockerB2.duration_p95', 'trace.from_dockerB2.row_count', 'trace.to_dockerA1.duration_mean', 'trace.to_dockerA1.duration_p95', 'trace.to_dockerA1.row_count', 'trace.to_dockerA2.duration_mean', 'trace.to_dockerA2.duration_p95', 'trace.to_dockerA2.row_count', 'trace.to_dockerB1.duration_mean', 'trace.to_dockerB1.duration_p95', 'trace.to_dockerB1.row_count', 'trace.to_dockerB2.duration_mean', 'trace.to_dockerB2.duration_p95']},    cmdb_id                       trace_name  count   p95_value
3     IG01         trace.self.duration_mean     30    7.269820
6     IG01  trace.to_Tomcat01.duration_mean     30    0.170048
5     IG01             trace.self.row_count     30  498.800000
4     IG01          trace.self.duration_p95     30   60.012457
8     IG01      trace.to_Tomcat01.row_count     30   63.000000
9     IG01  trace.to_Tomcat02.duration_mean     30    0.158716
10    IG01   trace.to_Tomcat02.duration_p95     30    0.982090
7     IG01   trace.to_Tomcat01.duration_p95     30    1.058635
11    IG01      trace.to_Tomcat02.row_count     30   66.100000
14    IG01      trace.to_Tomcat03.row_count     30   69.000000
13    IG01   trace.to_Tomcat03.duration_p95     30    0.990340
12    IG01  trace.to_Tomcat03.duration_mean     30    0.164210
20    IG02             trace.self.row_count     30  520.300000
21    IG02  trace.to_Tomcat01.duration_mean     30    0.150707
22    IG02   trace.to_Tomcat01.duration_p95     30    1.035510
15    IG01  trace.to_Tomcat04.duration_mean     30    0.151750
16    IG01   trace.to_Tomcat04.duration_p95     30    1.018820
17    IG01      trace.to_Tomcat04.row_count     30   66.500000
18    IG02         trace.self.duration_mean     30    6.833541
19    IG02          trace.self.duration_p95     30   61.013140, {'total_rows': 429, 'min_timestamp': 1615559400, 'min_timestamp_tz': '2021-03-12T22:30:00+08:00', 'max_timestamp': 1615561200, 'max_timestamp_tz': '2021-03-12T23:00:00+08:00', 'unique_cmdb_ids_sample (<=50)': ['Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'Mysql02'], 'unique_names_sample (<=50)': ['log.error_count', 'log.gc_pause_max', 'log.row_count']},      cmdb_id          log_name  count  p95_value
4   Tomcat01     log.row_count     31    121.500
2   Tomcat01   log.error_count     31      0.000
17  apache02     log.row_count     31    263.000
15  apache01     log.row_count     31    253.000
8   Tomcat03   log.error_count     31      0.000
16  apache02   log.error_count     31      8.500
14  apache01   log.error_count     31      8.500
13  Tomcat04     log.row_count     31    131.500
11  Tomcat04   log.error_count     31      0.000
10  Tomcat03     log.row_count     31    129.500
5   Tomcat02   log.error_count     30      0.000
7   Tomcat02     log.row_count     30    133.750
1    Mysql02     log.row_count     15    138.100
0    Mysql02   log.error_count     15      0.000
6   Tomcat02  log.gc_pause_max     10      0.080
3   Tomcat01  log.gc_pause_max      7      0.087
9   Tomcat03  log.gc_pause_max      6      0.080
12  Tomcat04  log.gc_pause_max      6      0.105, 71,      timestamp   cmdb_id                                            message               timestamp_tz
0   1615559702  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...  2021-03-12T22:35:02+08:00
1   1615559710  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...  2021-03-12T22:35:10+08:00
2   1615559714  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...  2021-03-12T22:35:14+08:00
3   1615559716  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...  2021-03-12T22:35:16+08:00
4   1615559716  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...  2021-03-12T22:35:16+08:00
5   1615559717  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...  2021-03-12T22:35:17+08:00
6   1615559720  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...  2021-03-12T22:35:20+08:00
7   1615559723  apache01  IPAddress "POST /UOCP/base/ServiceTest2.json H...  2021-03-12T22:35:23+08:00
8   1615559724  apache01  IPAddress "POST /UOCP/person/ServiceTest3.json...  2021-03-12T22:35:24+08:00
9   1615559729  apache01  IPAddress "POST /UOCP/base/ServiceTest8.json H...  2021-03-12T22:35:29+08:00
10  1615559730  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...  2021-03-12T22:35:30+08:00
11  1615559734  apache02  IPAddress "POST /UOCP/base/ServiceTest4.json H...  2021-03-12T22:35:34+08:00
12  1615559735  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ...  2021-03-12T22:35:35+08:00
13  1615559745  apache02  IPAddress "POST /UOCP/base/ServiceTest4.json H...  2021-03-12T22:35:45+08:00
14  1615559754  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...  2021-03-12T22:35:54+08:00
15  1615559755  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...  2021-03-12T22:35:55+08:00
16  1615559759  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...  2021-03-12T22:35:59+08:00
17  1615559760  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...  2021-03-12T22:36:00+08:00
18  1615559765  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...  2021-03-12T22:36:05+08:00
19  1615559770  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...  2021-03-12T22:36:10+08:00)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# Reuse df_metric from kernel (already loaded). Define allowed components and time window.
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
start_ts = 1615559400
end_ts = 1615561200

# 2) Calculate global p95 and p5 for each (cmdb_id,kpi_name) using the full series (only for allowed components)
metric_allowed_full = df_metric[df_metric['cmdb_id'].isin(allowed_components)]
thresholds_df = (
    metric_allowed_full
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(global_p95=lambda s: float(s.quantile(0.95)),
         global_p5 = lambda s: float(s.quantile(0.05)))
)

# 3) Filter metric.csv to the incident time window (and only allowed components)
window_df = metric_allowed_full[
    (metric_allowed_full['timestamp'] >= start_ts) &
    (metric_allowed_full['timestamp'] <= end_ts)
].copy()

# 4) For each pair, compute window stats and anomalies using the global thresholds
# Merge thresholds onto window rows so each row knows its global thresholds
window_with_thr = window_df.merge(thresholds_df, on=['cmdb_id','kpi_name'], how='left')

# Flags for anomalies (inclusive as requested)
window_with_thr['is_above'] = (window_with_thr['value'] >= window_with_thr['global_p95']).astype(int)
window_with_thr['is_below'] = (window_with_thr['value'] <= window_with_thr['global_p5']).astype(int)

# Aggregations within window for pairs that have points
window_stats = (
    window_with_thr
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(points_in_window=('value','count'),
         anomalies_above_p95=('is_above','sum'),
         anomalies_below_p5=('is_below','sum'),
         max_value_in_window=('value','max'),
         min_value_in_window=('value','min'))
)

# Earliest anomaly timestamp per pair (consider either above or below)
anoms = window_with_thr[(window_with_thr['is_above']==1) | (window_with_thr['is_below']==1)]
earliest_anom = (
    anoms
    .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_anom_ts'})
)

# 4 cont'd) Merge everything back onto the full thresholds list so pairs with zero window points are included
result = thresholds_df.merge(window_stats, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0 (no points in window), keep earliest_anom_ts as NaN when none
result['points_in_window'] = result['points_in_window'].fillna(0).astype(int)
result['anomalies_above_p95'] = result['anomalies_above_p95'].fillna(0).astype(int)
result['anomalies_below_p5'] = result['anomalies_below_p5'].fillna(0).astype(int)
# max/min remain NaN when no points in window

# Compute a total anomalies column for sorting
result['anomalies_total'] = result['anomalies_above_p95'] + result['anomalies_below_p5']

# 5) Prepare compact output: top 20 rows by anomalies_total desc
result_sorted = result.sort_values(['anomalies_total','points_in_window'], ascending=[False, False])

# Selected columns and order
out_cols = ['cmdb_id','kpi_name','global_p95','global_p5',
            'points_in_window','anomalies_above_p95','anomalies_below_p5',
            'earliest_anom_ts','max_value_in_window','min_value_in_window','anomalies_total']
result_out = result_sorted[out_cols].head(20)

# Summary counts
total_pairs_scanned = len(thresholds_df)
pairs_with_anomaly_count = int((result['anomalies_total'] > 0).sum())

# Display compact results (variables)
result_out, total_pairs_scanned, pairs_with_anomaly_count

```
Out[2]:
```
# Compose a concise plain-English summary using kernel-stored results
top_rows = result_out.head(5)
lines = []
for _, r in top_rows.iterrows():
    lines.append(
        f"{r['cmdb_id']} | {r['kpi_name']} | global_p95={r['global_p95']:.6g} "
        f"p5={r['global_p5']:.6g} | points_in_window={r['points_in_window']} "
        f"anomalies_total={r['anomalies_total']} earliest_anom_ts={int(r['earliest_anom_ts'])}"
    )

summary = (
    "Metric-level anomaly scan summary (incident window: 1615559400 to 1615561200 UTC+8):\n\n"
    f"- Total (cmdb_id,kpi_name) pairs scanned: {total_pairs_scanned}\n"
    f"- Pairs with ≥1 anomaly in window: {pairs_with_anomaly_count}\n\n"
    "Key observations:\n"
    "- Every scanned pair reported at least one anomaly in the window (counts above).\n"
    "- Many top series show global_p95 == global_p5 and max==min==that value, so every in-window point "
    "meets both >=p95 and <=p5 conditions. This leads to anomalies being counted both above and below for those series.\n\n"
    "Top example pairs (up to 5):\n"
    + "\n".join("  - " + l for l in lines)
    + "\n\n"
    "Recommendation: inspect series with identical p95/p5 and constant-valued metrics (likely steady/constant counters or clipping) "
    "and confirm whether inclusive anomaly rules (>= p95, <= p5) are intended or should be refined."
)

# Display the summary string
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name    global_p95     global_p5  points_in_window  anomalies_above_p95  anomalies_below_p5  earliest_anom_ts  max_value_in_window  min_value_in_window  anomalies_total
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09                31                   31                  31        1615559400         2.040136e+09         2.040136e+09               62
67      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00                31                   31                  31        1615559400         0.000000e+00         0.000000e+00               62
78      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00  1.000000e+00                31                   31                  31        1615559400         1.000000e+00         1.000000e+00               62
81      IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01  3.000000e-01                31                   31                  31        1615559400         3.000000e-01         3.000000e-01               62
82      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00  3.000000e+00                31                   31                  31        1615559400         3.000000e+00         3.000000e+00               62
90      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09                31                   31                  31        1615559400         2.040136e+09         2.040136e+09               62
168     IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00  1.000000e+00                31                   31                  31        1615559400         1.000000e+00         1.000000e+00               62
171     IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01  3.000000e-01                31                   31                  31        1615559400         3.000000e-01         3.000000e-01               62
172     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00  3.000000e+00                31                   31                  31        1615559400         3.000000e+00         3.000000e+00               62
180     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09                31                   31                  31        1615559400         2.040136e+09         2.040136e+09               62
183     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.069495e+08  5.069495e+08                31                   31                  31        1615559400         5.069495e+08         5.069495e+08               62
271     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09                31                   31                  31        1615559400         2.040136e+09         2.040136e+09               62
274     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  9.471521e+08  9.471521e+08                31                   31                  31        1615559400         9.471521e+08         9.471521e+08               62
574  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00                31                   31                  31        1615559400         0.000000e+00         0.000000e+00               62
575  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00  0.000000e+00                31                   31                  31        1615559400         0.000000e+00         0.000000e+00               62
577  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  0.000000e+00  0.000000e+00                31                   31                  31        1615559400         0.000000e+00         0.000000e+00               62
581  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  7.000000e+00  7.000000e+00                31                   31                  31        1615559400         7.000000e+00         7.000000e+00               62
800  Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00  0.000000e+00                31                   31                  31        1615559400         0.000000e+00         0.000000e+00               62
803  Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  0.000000e+00  0.000000e+00                31                   31                  31        1615559400         0.000000e+00         0.000000e+00               62
807  Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2.400000e+01  2.400000e+01                31                   31                  31        1615559400         2.400000e+01         2.400000e+01               62, 1636, 1636)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse df_trace from kernel
tz = pytz.timezone('Asia/Shanghai')

allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
start_ts = 1615559400
end_ts = 1615561200

# 1) Compute global p95 and p5 for each (cmdb_id, trace_name) across the full series (only allowed components)
trace_allowed_full = df_trace[df_trace['cmdb_id'].isin(allowed_components)]
thresholds_trace = (
    trace_allowed_full
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_p95=lambda s: float(s.quantile(0.95)),
         global_p5 = lambda s: float(s.quantile(0.05)))
)

# 2) Filter trace.csv to the incident window (and allowed components)
trace_window = trace_allowed_full[
    (trace_allowed_full['timestamp'] >= start_ts) &
    (trace_allowed_full['timestamp'] <= end_ts)
].copy()

# 3) Merge thresholds onto window rows so each row carries its global thresholds
trace_window = trace_window.merge(thresholds_trace, on=['cmdb_id','trace_name'], how='left')

# Flags for anomalies (inclusive)
trace_window['is_above'] = (trace_window['value'] >= trace_window['global_p95']).astype(int)
trace_window['is_below'] = (trace_window['value'] <= trace_window['global_p5']).astype(int)

# Aggregate window stats per pair
window_stats_trace = (
    trace_window
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(points_in_window=('value','count'),
         anomalies_above_p95=('is_above','sum'),
         anomalies_below_p5=('is_below','sum'),
         max_value_in_window=('value','max'),
         min_value_in_window=('value','min'))
)

# Earliest anomaly timestamp per pair (either above or below)
anoms_trace = trace_window[(trace_window['is_above']==1) | (trace_window['is_below']==1)]
earliest_anom_trace = (
    anoms_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_anom_ts'})
)

# 4) Combine with full thresholds so pairs with zero points in window are included
trace_result = thresholds_trace.merge(window_stats_trace, on=['cmdb_id','trace_name'], how='left')
trace_result = trace_result.merge(earliest_anom_trace, on=['cmdb_id','trace_name'], how='left')

# Fill counts with 0 where no points in window
trace_result['points_in_window'] = trace_result['points_in_window'].fillna(0).astype(int)
trace_result['anomalies_above_p95'] = trace_result['anomalies_above_p95'].fillna(0).astype(int)
trace_result['anomalies_below_p5'] = trace_result['anomalies_below_p5'].fillna(0).astype(int)
trace_result['anomalies_total'] = trace_result['anomalies_above_p95'] + trace_result['anomalies_below_p5']

# Keep max/min as-is (NaN when no points in window)

# 5) Prepare compact output: top 20 rows sorted by anomalies_total desc, then points_in_window desc
trace_result_sorted = trace_result.sort_values(['anomalies_total','points_in_window'], ascending=[False, False])

out_cols = ['cmdb_id','trace_name','global_p95','global_p5',
            'points_in_window','anomalies_above_p95','anomalies_below_p5','anomalies_total',
            'earliest_anom_ts','max_value_in_window','min_value_in_window']

trace_out = trace_result_sorted[out_cols].head(20)

total_pairs_scanned = len(thresholds_trace)
pairs_with_anomaly_count = int((trace_result['anomalies_total'] > 0).sum())

# Display compact results (variables)
trace_out, total_pairs_scanned, pairs_with_anomaly_count

```
Out[3]:
```
Summary of trace-level anomaly scan (incident window 1615559400 to 1615561200 UTC+8):

- Total (cmdb_id, trace_name) pairs scanned: 186
- Pairs with ≥1 anomaly in the window: 186 (every scanned pair had at least one anomaly by the inclusive threshold rules)

Top example pairs (from the top 20 by anomalies_total):
1. MG01 | trace.from_dockerB1.duration_p95 — global_p95=0.021977, global_p5=0.008000; points_in_window=30; anomalies_total=10 (2 above p95 + 8 below p5); earliest_anom_ts=1615559700; max=0.02475; min=0.0075
2. MG02 | trace.from_dockerB1.duration_p95 — global_p95=0.012377, global_p5=0.008000; points_in_window=30; anomalies_total=10; earliest_anom_ts=1615559580; max=0.0915; min=0.0080
3. MG02 | trace.from_dockerA2.row_count — global_p95=16.3, global_p5=4.0; points_in_window=28; anomalies_total=7; earliest_anom_ts=1615559400; max=17.0; min=3.0
4. IG01 | trace.to_Tomcat01.row_count — global_p95=63.0, global_p5=34.8; points_in_window=30; anomalies_total=6; earliest_anom_ts=1615559580; max=65; min=32
5. IG02 | trace.to_Tomcat02.row_count — global_p95=69.0, global_p5=28.45; points_in_window=30; anomalies_total=6; earliest_anom_ts=1615559400; max=73; min=28

Other observations from the top results:
- Many top anomalies are on row_count metrics (high/low counts) and some duration_p95 metrics (spikes in latency).
- Multiple components appear repeatedly (MG01, MG02, IG01, IG02, Tomcat01–03).
- Earliest anomaly timestamps for top items fall within the window (examples: 1615559400, 1615559460, 1615559580, 1615559700) — inspect those timepoints.
- Several series show multiple anomalies per series (anomalies_total up to 10 in the top rows shown).

Recommended next steps:
- Investigate the listed components (MG01, MG02, IG01/IG02, Tomcat nodes) and their trace.row_count and duration_p95 series around the earliest anomaly timestamps.
- Check whether the inclusive anomaly rule (value >= global_p95 or <= global_p5) is appropriate, since it flags both spikes and drops; refine thresholds if needed.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                        trace_name  global_p95  global_p5  points_in_window  anomalies_above_p95  anomalies_below_p5  anomalies_total  earliest_anom_ts  max_value_in_window  min_value_in_window
52       MG01  trace.from_dockerB1.duration_p95    0.021977   0.008000                30                    2                   8               10        1615559700             0.024750             0.007500
94       MG02  trace.from_dockerB1.duration_p95    0.012377   0.008000                30                    2                   8               10        1615559580             0.091500             0.008000
92       MG02     trace.from_dockerA2.row_count   16.300000   4.000000                28                    2                   5                7        1615559400            17.000000             3.000000
8        IG01       trace.to_Tomcat01.row_count   63.000000  34.800000                30                    4                   2                6        1615559580            65.000000            32.000000
26       IG02       trace.to_Tomcat02.row_count   69.000000  28.450000                30                    4                   2                6        1615559400            73.000000            28.000000
41       MG01     trace.from_Tomcat03.row_count   67.200000  16.000000                30                    2                   4                6        1615559460            77.000000            13.000000
88       MG02  trace.from_dockerA1.duration_p95    0.009885   0.006000                30                    2                   4                6        1615559460             0.012550             0.006000
119  Tomcat01         trace.from_IG01.row_count   63.000000  34.800000                30                    4                   2                6        1615559580            65.000000            32.000000
140  Tomcat02         trace.from_IG02.row_count   69.000000  28.450000                30                    4                   2                6        1615559400            73.000000            28.000000
164  Tomcat03           trace.to_MG01.row_count   67.200000  16.000000                30                    2                   4                6        1615559460            77.000000            13.000000
50       MG01     trace.from_dockerA2.row_count   18.000000   4.000000                29                    3                   3                6        1615559880            19.000000             3.000000
11       IG01       trace.to_Tomcat02.row_count   66.100000  27.000000                30                    2                   3                5        1615559520            71.000000            25.000000
14       IG01       trace.to_Tomcat03.row_count   69.000000  16.350000                30                    3                   2                5        1615559460            70.000000            14.000000
32       IG02       trace.to_Tomcat04.row_count   69.400000  40.000000                30                    2                   3                5        1615559400            74.000000            35.000000
35       MG01     trace.from_Tomcat01.row_count   68.650000  38.000000                30                    2                   3                5        1615559520            70.000000            32.000000
47       MG01     trace.from_dockerA1.row_count   18.100000   3.000000                30                    2                   3                5        1615559820            22.000000             1.000000
53       MG01     trace.from_dockerB1.row_count   66.950000  16.000000                30                    2                   3                5        1615559460            86.000000            16.000000
55       MG01  trace.from_dockerB2.duration_p95    0.205392   0.007000                30                    2                   3                5        1615559700             0.528000             0.007000
60       MG01          trace.self.duration_mean    0.079646   0.033028                30                    2                   3                5        1615559400             0.082787             0.029918
77       MG02     trace.from_Tomcat01.row_count   61.000000  33.900000                30                    3                   2                5        1615559460            63.000000            28.000000, 186, 186)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse kernel-stored dataframes: df_log, df_error (already loaded earlier)
candidate_cmdb = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04"]
start_ts = 1615559400
end_ts = 1615561200
tz = pytz.timezone('Asia/Shanghai')

# 2) Filter log.csv for candidates and time window
log_window = df_log[
    df_log['cmdb_id'].isin(candidate_cmdb) &
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts)
].copy()

# Aggregate per (cmdb_id, log_name)
# points_in_window, sum_value, max_value
log_agg = (
    log_window
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(points_in_window=('value','count'),
         sum_value=('value','sum'),
         max_value=('value','max'))
)

# earliest_ts_value_gt_0: minimum timestamp where value > 0
vals_gt0 = log_window[log_window['value'] > 0]
earliest_gt0 = (
    vals_gt0
    .groupby(['cmdb_id','log_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_ts_value_gt_0'})
)

log_agg = log_agg.merge(earliest_gt0, on=['cmdb_id','log_name'], how='left')

# Sort by sum_value desc and keep top 20
log_agg_top20 = log_agg.sort_values('sum_value', ascending=False).head(20)

# 3) Filter error_logs.csv for candidates and time window
error_window = df_error[
    df_error['cmdb_id'].isin(candidate_cmdb) &
    (df_error['timestamp'] >= start_ts) &
    (df_error['timestamp'] <= end_ts)
].copy()

total_error_rows = len(error_window)

# Per-cmdb_id counts and earliest error timestamp
error_per_cmdb = (
    error_window
    .groupby('cmdb_id', as_index=False)
    .agg(error_count=('message','count'),
         earliest_error_ts=('timestamp','min'))
    .sort_values('error_count', ascending=False)
)

# Prepare up to first 20 raw error rows with timezone string
def epoch_to_tz_iso(epoch):
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).isoformat()

if total_error_rows > 0:
    error_rows_head = error_window[['timestamp','cmdb_id','message']].head(20).copy()
    error_rows_head['timestamp_tz'] = error_rows_head['timestamp'].apply(lambda x: epoch_to_tz_iso(int(x)))
else:
    error_rows_head = pd.DataFrame(columns=['timestamp','cmdb_id','message','timestamp_tz'])

# Final compact outputs (variables to display)
log_agg_top20, total_error_rows, error_per_cmdb, error_rows_head

```
Out[4]:
```
```python
# Compose a concise plain-English summary using kernel-stored results
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

def epoch_to_tz_iso(epoch):
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).isoformat()

# Reuse variables from kernel: log_agg_top20, total_error_rows, error_per_cmdb, error_rows_head
top_logs = log_agg_top20.head(6).copy()
# format earliest_ts_value_gt_0 to readable or None
def fmt_ts(x):
    return epoch_to_tz_iso(int(x)) if pd.notna(x) else None

top_log_lines = []
for _, r in top_logs.iterrows():
    top_log_lines.append(
        f"{r['cmdb_id']} | {r['log_name']} | points={int(r['points_in_window'])} "
        f"sum={r['sum_value']:.0f} max={r['max_value']:.0f} earliest_value>0={fmt_ts(r.get('earliest_ts_value_gt_0'))}"
    )

# error summary per cmdb
err_summary_lines = []
for _, r in error_per_cmdb.iterrows():
    err_summary_lines.append(
        f"{r['cmdb_id']} | errors={int(r['error_count'])} earliest={epoch_to_tz_iso(int(r['earliest_error_ts']))}"
    )

summary_text = (
    "Log & error-log inspection (apache* and Tomcat nodes) for window 1615559400..1615561200 (UTC+8):\n\n"
    "Top log aggregates (by sum_value, up to 6 shown):\n"
    + "\n".join(" - " + l for l in top_log_lines) + "\n\n"
    f"Error logs: total_error_rows={total_error_rows} across candidate front-end/Tomcat nodes.\n"
    "Per-node error counts and earliest error time:\n"
    + "\n".join(" - " + l for l in err_summary_lines) + "\n\n"
    "Context from sample raw error rows: many Apache POST requests around 2021-03-12T22:35:00+08 (see sampled error rows).\n\n"
    "Key takeaways:\n"
    "- Row-count logs dominate activity: apache02 (sum 6106) and apache01 (sum 6002) are busiest, followed by Tomcat nodes.\n"
    "- Apache nodes produced most error-log entries (apache01:36, apache02:35) with earliest errors ~2021-03-12T22:35:02–22:35:14+08.\n"
    "- GC pause metrics exist on Tomcat nodes but with much smaller totals; some log.error_count for Tomcats are zero in this window.\n\n"
    "Recommendation: investigate Apache traffic spikes and corresponding Tomcat row_count peaks around 22:35+08; inspect the sampled error rows for request patterns and correlate with trace/metric anomalies."
)

# Display summary string (variable)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  points_in_window  sum_value  max_value  earliest_ts_value_gt_0
15  apache02     log.row_count                31    6106.00     266.00            1.615559e+09
13  apache01     log.row_count                31    6002.00     259.00            1.615559e+09
11  Tomcat04     log.row_count                31    3029.00     145.00            1.615559e+09
2   Tomcat01     log.row_count                31    3007.00     127.00            1.615559e+09
5   Tomcat02     log.row_count                30    3004.00     136.00            1.615559e+09
8   Tomcat03     log.row_count                31    2923.00     135.00            1.615559e+09
12  apache01   log.error_count                31      36.00       9.00            1.615560e+09
14  apache02   log.error_count                31      35.00      10.00            1.615560e+09
4   Tomcat02  log.gc_pause_max                10       0.63       0.08            1.615559e+09
10  Tomcat04  log.gc_pause_max                 6       0.39       0.11            1.615560e+09
1   Tomcat01  log.gc_pause_max                 7       0.38       0.09            1.615559e+09
7   Tomcat03  log.gc_pause_max                 6       0.31       0.08            1.615560e+09
6   Tomcat03   log.error_count                31       0.00       0.00                     NaN
3   Tomcat02   log.error_count                30       0.00       0.00                     NaN
0   Tomcat01   log.error_count                31       0.00       0.00                     NaN
9   Tomcat04   log.error_count                31       0.00       0.00                     NaN, 71,     cmdb_id  error_count  earliest_error_ts
0  apache01           36         1615559714
1  apache02           35         1615559702,      timestamp   cmdb_id                                            message               timestamp_tz
0   1615559702  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...  2021-03-12T22:35:02+08:00
1   1615559710  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...  2021-03-12T22:35:10+08:00
2   1615559714  apache01  IPAddress "POST /UOCP/base/ServiceTest11.json ...  2021-03-12T22:35:14+08:00
3   1615559716  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...  2021-03-12T22:35:16+08:00
4   1615559716  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...  2021-03-12T22:35:16+08:00
5   1615559717  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...  2021-03-12T22:35:17+08:00
6   1615559720  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...  2021-03-12T22:35:20+08:00
7   1615559723  apache01  IPAddress "POST /UOCP/base/ServiceTest2.json H...  2021-03-12T22:35:23+08:00
8   1615559724  apache01  IPAddress "POST /UOCP/person/ServiceTest3.json...  2021-03-12T22:35:24+08:00
9   1615559729  apache01  IPAddress "POST /UOCP/base/ServiceTest8.json H...  2021-03-12T22:35:29+08:00
10  1615559730  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...  2021-03-12T22:35:30+08:00
11  1615559734  apache02  IPAddress "POST /UOCP/base/ServiceTest4.json H...  2021-03-12T22:35:34+08:00
12  1615559735  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ...  2021-03-12T22:35:35+08:00
13  1615559745  apache02  IPAddress "POST /UOCP/base/ServiceTest4.json H...  2021-03-12T22:35:45+08:00
14  1615559754  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...  2021-03-12T22:35:54+08:00
15  1615559755  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...  2021-03-12T22:35:55+08:00
16  1615559759  apache02  IPAddress "POST /UOCP/person/ServiceTest10.jso...  2021-03-12T22:35:59+08:00
17  1615559760  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...  2021-03-12T22:36:00+08:00
18  1615559765  apache01  IPAddress "POST /UOCP/base/ServiceTest7.json H...  2021-03-12T22:36:05+08:00
19  1615559770  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...  2021-03-12T22:36:10+08:00)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse df_metric from kernel
tz = pytz.timezone('Asia/Shanghai')
component = "Tomcat01"
start_ts = 1615559400
end_ts = 1615561200

# 1) Select Tomcat01 rows
tomcat01_full = df_metric[df_metric['cmdb_id'] == component].copy()

# 2) Compute global p95 and p5 per (Tomcat01, kpi_name) using full series (no time filtering)
tomcat01_thresholds = (
    tomcat01_full
    .groupby(['kpi_name'], as_index=False)['value']
    .agg(global_p95=lambda s: float(s.quantile(0.95)),
         global_p5 = lambda s: float(s.quantile(0.05)))
)

# 3) Filter Tomcat01 rows to incident window
tomcat01_window = tomcat01_full[
    (tomcat01_full['timestamp'] >= start_ts) &
    (tomcat01_full['timestamp'] <= end_ts)
].copy()

# 4) Merge thresholds onto window rows and compute anomaly flags (inclusive)
tomcat01_window = tomcat01_window.merge(tomcat01_thresholds, on='kpi_name', how='left')
tomcat01_window['is_above'] = (tomcat01_window['value'] >= tomcat01_window['global_p95']).astype(int)
tomcat01_window['is_below'] = (tomcat01_window['value'] <= tomcat01_window['global_p5']).astype(int)

# Aggregate window stats per kpi_name
tomcat01_stats = (
    tomcat01_window
    .groupby('kpi_name', as_index=False)
    .agg(points_in_window=('value','count'),
         anomalies_above_p95=('is_above','sum'),
         anomalies_below_p5=('is_below','sum'),
         max_value_in_window=('value','max'),
         min_value_in_window=('value','min'))
)

# Earliest anomaly timestamp per kpi_name (either above or below)
anoms = tomcat01_window[(tomcat01_window['is_above']==1) | (tomcat01_window['is_below']==1)]
tomcat01_earliest = (
    anoms
    .groupby('kpi_name', as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_anom_ts'})
)

# 4 cont'd) Combine thresholds with stats to include KPIs that may have zero points in window
tomcat01_result = tomcat01_thresholds.merge(tomcat01_stats, on='kpi_name', how='left')
tomcat01_result = tomcat01_result.merge(tomcat01_earliest, on='kpi_name', how='left')

# Fill NaNs for counts with 0; leave max/min/earliest as NaN when no points/anoms
tomcat01_result['points_in_window'] = tomcat01_result['points_in_window'].fillna(0).astype(int)
tomcat01_result['anomalies_above_p95'] = tomcat01_result['anomalies_above_p95'].fillna(0).astype(int)
tomcat01_result['anomalies_below_p5'] = tomcat01_result['anomalies_below_p5'].fillna(0).astype(int)
tomcat01_result['anomalies_total'] = tomcat01_result['anomalies_above_p95'] + tomcat01_result['anomalies_below_p5']

# 5) Filter to KPI pairs with anomalies_total >= 1 and sort by earliest_anom_ts ascending
tomcat01_anoms = tomcat01_result[tomcat01_result['anomalies_total'] >= 1].copy()
tomcat01_anoms_sorted = tomcat01_anoms.sort_values('earliest_anom_ts', ascending=True)

# Select and order required columns
out_cols = ['kpi_name','global_p95','global_p5','points_in_window',
            'anomalies_above_p95','anomalies_below_p5','anomalies_total',
            'earliest_anom_ts','max_value_in_window','min_value_in_window']

tomcat01_out = tomcat01_anoms_sorted[out_cols].head(20)

# Counts
total_kpi_pairs_scanned = int(len(tomcat01_thresholds))
kpi_pairs_with_anom = int((tomcat01_result['anomalies_total'] > 0).sum())

# Display compact results
tomcat01_out, total_kpi_pairs_scanned, kpi_pairs_with_anom

```
Out[5]:
```
```python
# Plain-English summary of Tomcat01 metric-level fault localization (using kernel-stored results)
summary_text = (
    "Tomcat01 fault-localization summary:\n\n"
    "- Scope: examined Tomcat01 metrics (107 distinct kpi_name pairs).\n"
    "- Findings: every scanned KPI pair (107) showed ≥1 anomaly within the incident window (1615559400..1615561200 UTC+8).\n"
    "- Earliest anomalies: occur at the start of the window (earliest_anom_ts = 1615559400) for the top items.\n\n"
    "Top signals and interpretations:\n"
    "1) Memory pressure: OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc is at 98% (global p95/p5 = 98). "
    "All in-window points equal 98 → flagged as anomalies (21 above, 21 below by the inclusive rule). "
    "This indicates the JVM/host memory metric is saturated/constant near 98%.\n\n"
    "2) Tomcat session counters: many Tomcat-Sessions series (ActiveCounter, KeepaliveCounter, RejectedSessions, and several logHome/session metrics) are constant zero during the window "
    "(global p95/p5 = 0, max/min = 0) and thus were flagged heavily. Constant-zero session metrics suggest no tracked sessions or a metrics/reporting anomaly.\n\n"
    "3) Thread counts & limits: Tomcat-Threads series show CurrentThreads/MaxThreads values (e.g., CurrentThreads ~100, MaxThreads = 500) are constant and flagged due to equality with thresholds. "
    "This highlights thread usage at or near fixed values but not necessarily a dynamic spike.\n\n"
    "4) Requests and file metrics: Tomcat-Requests_RequestCount shows very large values (~2.426–2.429e6) with some anomaly points (anomalies_total small relative to points). A file-size/agent log metric also shows small fluctuations flagged as anomalies.\n\n"
    "5) Misc: memory submetrics (FreeMem, CacheMem, NoCacheMemPerc, UserMem) and a single-CPU util (CPU-3) show minor anomalies at the window start.\n\n"
    "Important caveat about these anomaly counts:\n"
    "- Many series are constant or clipped (global_p95 == global_p5 == value). Because the rule used is inclusive (value >= p95 OR <= p5), constant series get counted as both 'above' and 'below' simultaneously, inflating anomalies_total. "
    "This can produce high anomaly counts even when metrics are simply constant.\n\n"
    "Recommendations:\n"
    "1) Investigate high host/JVM memory (98%) on Tomcat01 immediately — this is the most actionable resource-level symptom.\n"
    "2) Check Tomcat session reporting and business logic: constant-zero session counters vs. high request counts is suspicious (either no session persistence or metric collection issue).\n"
    "3) Correlate the earliest anomaly time (1615559400 → 2021-03-12T22:30:00+08) with traces/logs around that time to find causal events.\n"
    "4) Refine anomaly rule to avoid double-counting (use exclusive comparisons or separate logic for constant-valued series) to reduce false positives from clipped/constant metrics.\n\n"
    "If you want, I can (a) show the full list of Tomcat01 KPI pairs with their earliest anomaly timestamps, (b) extract raw metric time series for the top 5 signals, or (c) correlate these findings with Tomcat logs/traces."
)

# Display the summary (variable, as required)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                              kpi_name    global_p95     global_p5  points_in_window  anomalies_above_p95  anomalies_below_p5  anomalies_total  earliest_anom_ts  max_value_in_window  min_value_in_window
6                      OSLinux-CPU_CPU-3_SingleCpuUtil  1.803230e+00  1.055975e+00                12                    1                   1                2        1615559400         1.831500e+00         9.644000e-01
62        OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  4.986520e+01  4.982670e+01                20                    5                   6               11        1615559400         4.986520e+01         4.982670e+01
61        OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01  9.800000e+01                21                   21                  21               42        1615559400         9.800000e+01         9.800000e+01
63               OSLinux-OSLinux_MEMORY_MEMORY_UserMem  7.665450e+03  7.643550e+03                18                    1                   1                2        1615559400         7.668000e+03         7.641000e+03
59            OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem  1.446000e+02  1.234000e+02                18                    1                   1                2        1615559400         1.480000e+02         1.200000e+02
39   OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...  3.532000e-01  3.525400e-01                18                    4                   1                5        1615559400         3.532000e-01         3.522000e-01
58              OSLinux-OSLinux_MEMORY_MEMORY_CacheMem  3.781600e+03  3.762550e+03                18                    1                   1                2        1615559400         3.785000e+03         3.760000e+03
98         Tomcat-Sessions_7441--_SESSIONActiveCounter  0.000000e+00  0.000000e+00                31                   31                  31               62        1615559400         0.000000e+00         0.000000e+00
104  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02  1.000000e+02                31                   31                  31               62        1615559400         1.000000e+02         1.000000e+02
105  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  0.000000e+00  0.000000e+00                31                   31                  30               61        1615559400         1.000000e+00         0.000000e+00
106  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  5.000000e+02                31                   31                  31               62        1615559400         5.000000e+02         5.000000e+02
99      Tomcat-Sessions_7441--_SESSIONKeepaliveCounter  0.000000e+00  0.000000e+00                31                   31                  31               62        1615559400         0.000000e+00         0.000000e+00
100     Tomcat-Sessions_7441--_SESSIONRejectedSessions  0.000000e+00  0.000000e+00                31                   31                  31               62        1615559400         0.000000e+00         0.000000e+00
101  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  0.000000e+00  0.000000e+00                31                   31                  31               62        1615559400         0.000000e+00         0.000000e+00
102  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  0.000000e+00  0.000000e+00                31                   31                  31               62        1615559400         0.000000e+00         0.000000e+00
103  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  0.000000e+00  0.000000e+00                31                   31                  31               62        1615559400         0.000000e+00         0.000000e+00
96   Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...  0.000000e+00  0.000000e+00                31                   31                  31               62        1615559400         0.000000e+00         0.000000e+00
97   Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...  0.000000e+00  0.000000e+00                31                   31                  31               62        1615559400         0.000000e+00         0.000000e+00
94   Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.429628e+06  2.426972e+06                31                    2                   2                4        1615559400         2.429770e+06         2.426795e+06
85                  OSLinux-OSLinux_ZABBIX_Host_Uptime  6.760098e+06  6.758544e+06                18                    1                   1                2        1615559400         6.760149e+06         6.758440e+06, 107, 107)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse kernel-stored df_metric
component = "Tomcat01"
start_ts = 1615559400
end_ts = 1615561200

# 1) Select Tomcat01 rows with requested substrings (case-sensitive)
pattern = r"Tomcat-Requests|Request|Processing|MaxTime"
tomcat_selected = df_metric[
    (df_metric['cmdb_id'] == component) &
    (df_metric['kpi_name'].str.contains(pattern, na=False))
].copy()

# 2) Compute global p95 and p5 per kpi_name using full Tomcat01 selected series (no time filter)
thresholds_requests = (
    tomcat_selected
    .groupby('kpi_name', as_index=False)['value']
    .agg(global_p95=lambda s: float(s.quantile(0.95)),
         global_p5 = lambda s: float(s.quantile(0.05)))
)

# 3) Filter Tomcat01 selected rows to the incident window
tomcat_requests_window = tomcat_selected[
    (tomcat_selected['timestamp'] >= start_ts) &
    (tomcat_selected['timestamp'] <= end_ts)
].copy()

# 4) Merge thresholds onto window rows and compute anomaly flags (inclusive)
tomcat_requests_window = tomcat_requests_window.merge(thresholds_requests, on='kpi_name', how='left')
tomcat_requests_window['is_above'] = (tomcat_requests_window['value'] >= tomcat_requests_window['global_p95']).astype(int)
tomcat_requests_window['is_below'] = (tomcat_requests_window['value'] <= tomcat_requests_window['global_p5']).astype(int)

# Aggregate per kpi_name within window
agg_window_req = (
    tomcat_requests_window
    .groupby('kpi_name', as_index=False)
    .agg(points_in_window=('value','count'),
         anomalies_above_p95=('is_above','sum'),
         anomalies_below_p5=('is_below','sum'),
         max_value_in_window=('value','max'),
         mean_value_in_window=('value','mean'))
)

# Earliest anomaly timestamp per kpi_name (either above or below)
anoms_req = tomcat_requests_window[(tomcat_requests_window['is_above']==1) | (tomcat_requests_window['is_below']==1)]
earliest_anom_req = (
    anoms_req
    .groupby('kpi_name', as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_anom_ts'})
)

# 4 cont'd) Combine thresholds with aggregated window stats (to include KPIs possibly missing in window)
req_result = thresholds_requests.merge(agg_window_req, on='kpi_name', how='left')
req_result = req_result.merge(earliest_anom_req, on='kpi_name', how='left')

# Fill NaNs for counts with 0; keep earliest/max/mean as NaN when no points
req_result['points_in_window'] = req_result['points_in_window'].fillna(0).astype(int)
req_result['anomalies_above_p95'] = req_result['anomalies_above_p95'].fillna(0).astype(int)
req_result['anomalies_below_p5'] = req_result['anomalies_below_p5'].fillna(0).astype(int)
req_result['anomalies_total'] = req_result['anomalies_above_p95'] + req_result['anomalies_below_p5']

# 4 done: compute mean already included; ensure max present (NaN if no points)
# 5) Sort by anomalies_above_p95 desc then earliest_anom_ts asc and prepare final compact table
req_result_sorted = req_result.sort_values(['anomalies_above_p95','earliest_anom_ts'], ascending=[False, True])

out_cols = ['kpi_name','global_p95','global_p5','points_in_window',
            'anomalies_above_p95','anomalies_below_p5','anomalies_total',
            'earliest_anom_ts','max_value_in_window','mean_value_in_window']

tomcat_requests_out = req_result_sorted[out_cols].head(20)

# Display compact result (variable)
tomcat_requests_out

```
Out[6]:
```
Summary — Tomcat01 request-processing KPIs (incident window 2021-03-12 22:30:00+08 to 23:00:00+08):

Key rows found (compact):
- Tomcat-Requests_..._ErrorCount: global_p95 = 987.0, global_p5 = 987.0; 31 points in window; anomalies_above_p95 = 31, anomalies_below_p5 = 31 (anomalies_total = 62); earliest anomaly = 1615559400. max = mean = 987.0.
- Tomcat-Requests_..._MaxTimeResponse: global_p95 = 405396.0, global_p5 = 405396.0; 31 points; anomalies_above_p95 = 31, anomalies_below_p5 = 31 (total = 62); earliest = 1615559400. max = mean = 405396.0.
- Tomcat-Requests_..._Processing (processing time): global_p95 ≈ 1.579615e9, global_p5 ≈ 1.579362e9; 31 points; anomalies_above = 2, anomalies_below = 2 (total = 4); earliest = 1615559400; max ≈ 1.579626e9; mean ≈ 1.579496e9.
- Tomcat-Requests_..._RequestCount: global_p95 ≈ 2.429628e6, global_p5 ≈ 2.426972e6; 31 points; anomalies_above = 2, anomalies_below = 2 (total = 4); earliest = 1615559400; max ≈ 2.42977e6; mean ≈ 2.428307e6.

Interpretation and recommendations:
- Earliest anomalies for these request-related KPIs occur at the start of the window (1615559400). Investigate traces/logs at that timestamp first.
- Two KPIs (ErrorCount and MaxTimeResponse) are constant at their threshold (global_p95 == global_p5 == value), so every point is flagged both above and below under the inclusive rule. This causes inflated anomaly counts — treat these constant/clipped metrics carefully (likely a metric clipping or steady high error/time condition).
- RequestCount and Processing time show a small number of true deviations (2 points above p95 and 2 below p5 each). These are actionable and worth inspecting for specific spikes/drops.
- Action items: check Tomcat01 logs/traces around 2021-03-12T22:30:00+08, focus on the sources of high ErrorCount and MaxTimeResponse and the few processing/request spikes; also verify whether the constant-valued metrics are valid or indicate collection/clipping issues.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name    global_p95     global_p5  points_in_window  anomalies_above_p95  anomalies_below_p5  anomalies_total  earliest_anom_ts  max_value_in_window  mean_value_in_window
0  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  9.870000e+02  9.870000e+02                31                   31                  31               62        1615559400         9.870000e+02          9.870000e+02
1  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  4.053960e+05  4.053960e+05                31                   31                  31               62        1615559400         4.053960e+05          4.053960e+05
2  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.579615e+09  1.579362e+09                31                    2                   2                4        1615559400         1.579626e+09          1.579496e+09
3  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.429628e+06  2.426972e+06                31                    2                   2                4        1615559400         2.429770e+06          2.428307e+06```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse kernel-stored df_trace
start_ts = 1615559400
end_ts = 1615561200
tz = pytz.timezone('Asia/Shanghai')

# 1) Define requested pairs
requested_pairs = [
    ("IG01", "trace.to_Tomcat01.row_count"),
    ("IG01", "trace.to_Tomcat01.duration_mean"),
    ("IG01", "trace.to_Tomcat01.duration_p95"),
    ("Tomcat01", "trace.from_IG01.row_count"),
    ("Tomcat01", "trace.from_IG01.duration_mean"),
]
req_df = pd.DataFrame(requested_pairs, columns=['cmdb_id','trace_name'])

# Helper: mark rows belonging to requested pairs
df_trace_pairs_mask = df_trace.set_index(['cmdb_id','trace_name']).index.isin(requested_pairs)
trace_requested_full = df_trace[df_trace_pairs_mask].copy()

# 2) Global thresholds (full series, no time filtering) for present pairs
if not trace_requested_full.empty:
    thresholds = (
        trace_requested_full
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(global_p95=lambda s: float(s.quantile(0.95)),
             global_p5 = lambda s: float(s.quantile(0.05)))
    )
else:
    thresholds = pd.DataFrame(columns=['cmdb_id','trace_name','global_p95','global_p5'])

# 3) Filter to incident window for the requested pairs
trace_window = trace_requested_full[
    (trace_requested_full['timestamp'] >= start_ts) &
    (trace_requested_full['timestamp'] <= end_ts)
].copy()

# 4) Compute window-level anomaly stats
if not trace_window.empty:
    trace_window = trace_window.merge(thresholds, on=['cmdb_id','trace_name'], how='left')
    trace_window['is_above'] = (trace_window['value'] >= trace_window['global_p95']).astype(int)
    trace_window['is_below'] = (trace_window['value'] <= trace_window['global_p5']).astype(int)

    window_stats = (
        trace_window
        .groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(points_in_window=('value','count'),
             anomalies_above_p95=('is_above','sum'),
             anomalies_below_p5=('is_below','sum'),
             max_value_in_window=('value','max'),
             mean_value_in_window=('value','mean'))
    )

    anoms = trace_window[(trace_window['is_above']==1) | (trace_window['is_below']==1)]
    earliest_anom = (
        anoms
        .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anom_ts'})
    )
else:
    window_stats = pd.DataFrame(columns=['cmdb_id','trace_name','points_in_window',
                                         'anomalies_above_p95','anomalies_below_p5',
                                         'max_value_in_window','mean_value_in_window'])
    earliest_anom = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anom_ts'])

# Combine thresholds with window stats and earliest anomaly (only for pairs present in thresholds)
result = thresholds.merge(window_stats, on=['cmdb_id','trace_name'], how='left')
result = result.merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

# Fill numeric counts with 0 where appropriate; keep earliest/max/mean as NaN when no points
result['points_in_window'] = result['points_in_window'].fillna(0).astype(int)
result['anomalies_above_p95'] = result['anomalies_above_p95'].fillna(0).astype(int)
result['anomalies_below_p5'] = result['anomalies_below_p5'].fillna(0).astype(int)
result['anomalies_total'] = result['anomalies_above_p95'] + result['anomalies_below_p5']

# 5) Limit to the five requested pairs (but only those present) and keep compact
# Order by anomalies_total desc (then earliest_anom_ts asc) for readability
result_out = result.sort_values(['anomalies_total','earliest_anom_ts'], ascending=[False, True])[
    ['cmdb_id','trace_name','global_p95','global_p5','points_in_window',
     'anomalies_above_p95','anomalies_below_p5','anomalies_total',
     'earliest_anom_ts','max_value_in_window','mean_value_in_window']
]

# Counts for context
total_pairs_present = len(result_out)
pairs_with_anomaly = int((result_out['anomalies_total'] > 0).sum())

# Final compact outputs (variables to display)
result_out, total_pairs_present, pairs_with_anomaly

```
Out[7]:
```
```python
# Compose a concise plain-English summary for the IG01 -> Tomcat01 trace-series analysis
import pandas as pd
import pytz

# Reuse kernel-stored variables: result_out, total_pairs_present, pairs_with_anomaly
tz = pytz.timezone('Asia/Shanghai')

def epoch_to_tz_iso(epoch):
    return pd.to_datetime(int(epoch), unit='s', utc=True).tz_convert(tz).isoformat()

rows = []
for _, r in result_out.iterrows():
    earliest = (epoch_to_tz_iso(int(r['earliest_anom_ts'])) 
                if pd.notna(r['earliest_anom_ts']) else "None")
    rows.append(
        f"{r['cmdb_id']} | {r['trace_name']} | points={int(r['points_in_window'])} "
        f"anomalies_total={int(r['anomalies_total'])} (above={int(r['anomalies_above_p95'])}, below={int(r['anomalies_below_p5'])}) "
        f"earliest={earliest} max={r['max_value_in_window']:.6g} mean={r['mean_value_in_window']:.6g}"
    )

summary_text = (
    "IG01 → Tomcat01 trace-series anomaly summary (incident window 1615559400..1615561200 UTC+8):\n\n"
    f"- Pairs present: {total_pairs_present}; pairs with ≥1 anomaly: {pairs_with_anomaly}.\n\n"
    "Per-pair compact results:\n" + "\n".join(" - " + line for line in rows) + "\n\n"
    "Key takeaways:\n"
    "- Both sides of the call chain (IG01 -> Tomcat01 and Tomcat01 <- IG01) show matching anomaly patterns for row_count and duration_mean.\n"
    "- Row_count series: 30 points in window, anomalies_total=6 (4 above p95, 2 below p5), earliest anomaly ~ 2021-03-12T22:33:00+08 (1615559580). max ~65, mean ~50.57.\n"
    "- Duration_mean series: 30 points, anomalies_total=4 (2 above, 2 below), earliest anomaly ~ 2021-03-12T22:31:00+08 (1615559460). max ~0.2387s, mean ~0.0979s.\n"
    "- duration_p95 at IG01 shows 4 anomalies (2 above, 2 below) with earliest at 1615559640; Tomcat01 duration_p95 was not present in the requested set.\n\n"
    "Recommendation: investigate the shared timestamps (especially 22:31 and 22:33+08) — both IG01 and Tomcat01 show synchronized anomalies in request counts and latencies, indicating a likely service-level issue in this call path. Correlate these times with logs and metrics (Tomcat requests/errors) for root cause."
)

# Display the summary string (variable)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                       trace_name  global_p95  global_p5  points_in_window  anomalies_above_p95  anomalies_below_p5  anomalies_total  earliest_anom_ts  max_value_in_window  mean_value_in_window
2      IG01      trace.to_Tomcat01.row_count   63.000000  34.800000                30                    4                   2                6        1615559580            65.000000             50.566667
4  Tomcat01        trace.from_IG01.row_count   63.000000  34.800000                30                    4                   2                6        1615559580            65.000000             50.566667
0      IG01  trace.to_Tomcat01.duration_mean    0.170048   0.047435                30                    2                   2                4        1615559460             0.238683              0.097896
3  Tomcat01    trace.from_IG01.duration_mean    0.170048   0.047435                30                    2                   2                4        1615559460             0.238683              0.097896
1      IG01   trace.to_Tomcat01.duration_p95    1.058635   0.067123                30                    2                   2                4        1615559640             1.080800              0.480978, 5, 5)```
```